### Import libraries

🌐 **Requests Library**: Used for sending HTTP requests.

📊 **Pandas Library**: Essential for data manipulation and analysis. Offers data structures like DataFrames.

🖨️ **Pretty Print Module**: Enhances the readability of output and formats data structures in a more user-friendly way.

🕷️ **Scrapy**: Used for web crawling and scraping. Extracts data from websites efficiently.

In [1]:
import requests      
import pandas as pd          
from pprint import pprint    
from scrapy import Selector   

### 🌍 Web Scraping the TfL Timetables:

🚇 Here we parse in the url to retrieve timetable information from TfL's official website.

First, we use the `requests` library to send an HTTP GET request 🎯

📡 We include a 'User-Agent' to identify the purpose of the scraping and a contact email

Stores the response from the server, which can be further processed to extract required information🧑‍💻

In [2]:
my_url = 'https://tfl.gov.uk/travel-information/timetables/'
headers = {'User-Agent': 'webscraping practice (p.ge@lse.ac.uk)'}
response = requests.get(my_url, headers = headers)
response

<Response [200]>

### 🕸️ Parsing and Extracting Links from TfL Response:

📄 Here, we use `Selector` from Scrapy to parse the HTML content from the previous response.

Then, we identify all relevant links using a CSS selector, targeting 'div#list-loader-data-bus.list-loader-data a' and extracting their 'href' attributes 🔗

🌐 If a link does not start with the base URL "https://tfl.gov.uk", we prepend it to make the link absolute.

In [3]:
sel = Selector(text=response.text)
links = sel.css('div#list-loader-data-bus.list-loader-data a::attr(href)').getall()

base_url = "https://tfl.gov.uk"
absolute_links = [base_url + link if not link.startswith(base_url) else link for link in links]

### 🧐 Filtering Specific Links from the Collected URLs:

📋 Define a list of `desired_elements` which contains specific parts of URLs we are interested in. These elements represent specific bus routes we are lookinig for at the 3 stations.

🔗 Using a list comprehension, we filter `absolute_links`. We only keep those links that contain any of the elements from our `desired_elements` list.

💡 This process results in `selected_links`, a refined list of URLs. Each URL in this list corresponds to one of the specified bus routes or services, ready for further analysis or data extraction.

In [4]:
desired_elements = ['/1/','/59/','/68/','/91/','/188/','/243/',
                    '/n1/','/n68/','/n91/','/n171/','/sl6/','/9/',
                    '/23/','/87/','/172/', '/n9/','/n44/','/n87/','/n155/']

selected_links = [link for link in absolute_links if any(elem in link for elem in desired_elements)]

In [5]:
url = 'https://tfl.gov.uk/bus/timetable/1?SelectedDate=mondaytothursday&LineId=1&Mode=bus&fromId=490000112M&direction=outbound'

response = requests.get(url)
response_text = response.text

sel = Selector(text=response_text)

timetable_data = []

for interval in sel.css('ul.timetable-list > li'):
    
    time_range = interval.css('strong::text').get()
    if time_range:
        time_range = time_range.strip()
    
    times = interval.css('ul.times li::text').getall()
    times = [time.strip() for time in times if time.strip()]
    
    if times:
        for time in times:
            timetable_data.append((time_range, time))
    else:
        
        timetable_data.append((time_range,))

df = pd.DataFrame(timetable_data, columns=['Time Range', 'Time'])
print(df)

           Time Range   Time
0   First Bus - 05:25   None
1               05:41  05:41
2               05:41  05:58
3               06:00  06:15
4               06:00  06:32
5               06:00  06:49
6               07:00  07:04
7               07:00  07:19
8               07:00  07:30
9               07:00  07:40
10              07:00  07:50
11              08:00   None
12              21:00   None
13              00:00  00:11
14   Last Bus - 00:24   None
